[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/reto_expres/Reto_expres_plantilla.ipynb)

# Reto exprés · Diseña, entrena y defiende tu red de visión

Un único cuaderno para los 6 problemas. **Solo se tocan las celdas ✏️** (cambiar un valor y pulsar ▶). Las ⚙️ se ejecutan tal cual.

- ✏️ **1** · elegir problema (1 valor)
- ✏️ **2** · comparar variantes de red y elegir **mirando solo la validación**
- ✏️ **3** · ver dónde se equivoca (la parte más interesante)
- ✏️ **4** · probar la aplicación con una imagen nueva
- Al final sale una **FICHA** con todos los números para vuestra presentación.

### ⚙️ Preparación — pulsa ▶

In [ ]:
#@title ⚙️ Ejecuta esta celda primero
import os, time
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)
print("Listo ✔")

### ⚙️ Generador de imágenes del curso (el mismo del reto S11) — pulsa ▶

Solo lo usan los problemas `alojamientos` y `formas`. No hace falta leerlo.

In [ ]:
from PIL import Image, ImageDraw

CLASES_ALOJ = ["hotel", "apartamentos", "casa_rural", "camping", "piscina"]
CLASES_FORMAS = ["rayas_horizontales", "rayas_verticales", "diagonales", "cuadricula", "puntos", "circulo", "rectangulo", "triangulo"]


def _color(rng, base, var=30):
    """Color aleatorio cercano a `base` (tupla RGB)."""
    return tuple(int(np.clip(c + rng.integers(-var, var + 1), 0, 255)) for c in base)


def _final(img, rng, ruido=8):
    """Convierte a uint8 con un poco de ruido y variación de brillo."""
    a = np.asarray(img, dtype="float32") * rng.uniform(0.85, 1.15)
    a = a + rng.normal(0, ruido, a.shape)
    return np.clip(a, 0, 255).astype("uint8")


def _ventanas(d, x0, y0, x1, y1, filas, cols, color, margen):
    """Cuadrícula de ventanas dentro del rectángulo (x0, y0, x1, y1)."""
    ancho, alto = (x1 - x0) / cols, (y1 - y0) / filas
    for f in range(filas):
        for c in range(cols):
            a, b = x0 + c * ancho + margen * ancho, y0 + f * alto + margen * alto
            d.rectangle([a, b, a + ancho * (1 - 2 * margen), b + alto * (1 - 2 * margen)], fill=color)


def dibujar_alojamiento(clase, rng, tam=48, ruido=8):
    """Dibujo esquemático (tam×tam×3, uint8) de un tipo de alojamiento. Datos SINTÉTICOS, no fotos reales."""
    E = 3                                   # supermuestreo para suavizar bordes
    S = tam * E
    img = Image.new("RGB", (S, S), _color(rng, (170, 205, 235), 25))          # cielo
    d = ImageDraw.Draw(img)
    if rng.random() < 0.5:                                                    # sol opcional
        cx, cy, r = rng.uniform(0.1, 0.9) * S, rng.uniform(0.08, 0.25) * S, rng.uniform(0.05, 0.08) * S
        d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(250, 235, 120))
    hor = rng.uniform(0.62, 0.72) * S                                         # línea del horizonte
    verde = _color(rng, (120, 170, 90), 25)
    if clase == "piscina":
        d.rectangle([0, hor * 0.6, S, S], fill=_color(rng, (225, 210, 170), 15))   # terraza
        x0, x1 = rng.uniform(0.08, 0.2) * S, rng.uniform(0.75, 0.92) * S
        y0, y1 = rng.uniform(0.5, 0.58) * S, rng.uniform(0.82, 0.94) * S
        d.rectangle([x0 - 3, y0 - 3, x1 + 3, y1 + 3], fill=(245, 245, 245))
        d.rectangle([x0, y0, x1, y1], fill=_color(rng, (60, 150, 210), 20))
        for k in range(3):
            yy = y0 + (k + 1) * (y1 - y0) / 4
            d.line([x0 + 6, yy, x1 - 6, yy], fill=(150, 210, 240), width=2)
        for k in range(2):                                                    # tumbonas
            tx = rng.uniform(0.1, 0.8) * S
            d.rectangle([tx, hor * 0.72, tx + 0.1 * S, hor * 0.72 + 0.04 * S], fill=_color(rng, (240, 120, 60), 30))
    else:
        d.rectangle([0, hor, S, S], fill=verde)                                # césped
    if clase == "hotel":
        w, h = rng.uniform(0.30, 0.42) * S, rng.uniform(0.5, 0.66) * S
        x0 = rng.uniform(0.1 * S, 0.9 * S - w); y0 = hor - h + 0.05 * S
        d.rectangle([x0, y0, x0 + w, hor + 0.05 * S], fill=_color(rng, (220, 200, 170)))
        _ventanas(d, x0, y0 + 0.02 * S, x0 + w, hor, int(rng.integers(4, 6)), 3, (60, 90, 140), 0.22)
    elif clase == "apartamentos":
        w, h = rng.uniform(0.7, 0.86) * S, rng.uniform(0.28, 0.38) * S
        x0 = rng.uniform(0.02 * S, 0.98 * S - w); y0 = hor - h + 0.05 * S
        d.rectangle([x0, y0, x0 + w, hor + 0.05 * S], fill=_color(rng, (235, 225, 205)))
        _ventanas(d, x0, y0, x0 + w, hor + 0.05 * S, 2, int(rng.integers(5, 7)), (70, 100, 150), 0.25)
        for k in (1, 2):                                                       # balcones (líneas horizontales)
            yy = y0 + k * (hor + 0.05 * S - y0) / 2
            d.line([x0, yy, x0 + w, yy], fill=(120, 110, 100), width=2)
    elif clase == "casa_rural":
        w, h = rng.uniform(0.32, 0.44) * S, rng.uniform(0.2, 0.28) * S
        x0 = rng.uniform(0.1 * S, 0.9 * S - w); y0 = hor - h + 0.05 * S
        d.rectangle([x0, y0, x0 + w, hor + 0.05 * S], fill=_color(rng, (235, 225, 200)))
        d.polygon([(x0 - 0.04 * S, y0), (x0 + w + 0.04 * S, y0), (x0 + w / 2, y0 - rng.uniform(0.14, 0.2) * S)],
                  fill=_color(rng, (170, 70, 50), 25))
        d.rectangle([x0 + w * 0.42, y0 + h * 0.4, x0 + w * 0.58, hor + 0.05 * S], fill=(110, 70, 40))
        d.rectangle([x0 + w * 0.1, y0 + h * 0.2, x0 + w * 0.3, y0 + h * 0.5], fill=(70, 100, 150))
    elif clase == "camping":
        d.rectangle([0, hor - 0.08 * S, S, S], fill=verde)
        w, h = rng.uniform(0.34, 0.5) * S, rng.uniform(0.28, 0.4) * S
        x0 = rng.uniform(0.05 * S, 0.95 * S - w); base = hor + 0.1 * S
        c = _color(rng, [(240, 140, 40), (60, 120, 200), (230, 200, 60)][int(rng.integers(0, 3))], 20)
        d.polygon([(x0, base), (x0 + w, base), (x0 + w / 2, base - h)], fill=c)
        d.polygon([(x0 + w * 0.4, base), (x0 + w * 0.6, base), (x0 + w / 2, base - h * 0.5)], fill=(50, 40, 40))
    if clase != "piscina" and rng.random() < 0.6:                              # árbol de adorno
        tx = rng.choice([rng.uniform(0.03, 0.12), rng.uniform(0.88, 0.97)]) * S
        d.rectangle([tx - 2, hor - 0.02 * S, tx + 2, hor + 0.12 * S], fill=(100, 70, 40))
        d.ellipse([tx - 0.07 * S, hor - 0.16 * S, tx + 0.07 * S, hor + 0.01 * S], fill=(50, 120, 60))
    return _final(img.resize((tam, tam), Image.LANCZOS), rng, ruido)


def dibujar_forma(clase, rng, tam=48):
    """Formas y texturas abstractas (tarea auxiliar de preentrenamiento). Devuelve tam×tam×3 uint8."""
    fondo, tinta = _color(rng, (rng.integers(20, 235),) * 3, 20), _color(rng, tuple(int(v) for v in rng.integers(0, 256, 3)), 10)
    if np.abs(np.array(fondo, dtype=int) - np.array(tinta, dtype=int)).sum() < 150:      # asegura contraste
        tinta = tuple(255 - c for c in fondo)
    E = 2
    S = tam * E
    img = Image.new("RGB", (S, S), fondo)
    d = ImageDraw.Draw(img)
    paso = int(rng.integers(6, 12)) * E
    off = int(rng.integers(0, paso))
    if clase == "rayas_horizontales":
        for y in range(off, S, paso): d.rectangle([0, y, S, y + paso // 2], fill=tinta)
    elif clase == "rayas_verticales":
        for x in range(off, S, paso): d.rectangle([x, 0, x + paso // 2, S], fill=tinta)
    elif clase == "diagonales":
        for k in range(-S, S, paso): d.line([k + off, 0, k + off + S, S], fill=tinta, width=max(2, paso // 3))
    elif clase == "cuadricula":
        for y in range(off, S, paso): d.line([0, y, S, y], fill=tinta, width=2)
        for x in range(off, S, paso): d.line([x, 0, x, S], fill=tinta, width=2)
    elif clase == "puntos":
        r = max(2, paso // 4)
        for y in range(off, S, paso):
            for x in range(off, S, paso): d.ellipse([x - r, y - r, x + r, y + r], fill=tinta)
    else:
        r = rng.uniform(0.18, 0.38) * S
        cx, cy = rng.uniform(r, S - r), rng.uniform(r, S - r)
        if clase == "circulo": d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=tinta)
        elif clase == "rectangulo": d.rectangle([cx - r, cy - r * rng.uniform(0.5, 1), cx + r, cy + r * rng.uniform(0.5, 1)], fill=tinta)
        else: d.polygon([(cx - r, cy + r), (cx + r, cy + r), (cx, cy - r)], fill=tinta)
    return _final(img.resize((tam, tam), Image.LANCZOS), rng)


def generar_lote(funcion, clases, n_por_clase, semilla=42, tam=48):
    """Devuelve (X uint8 [n, tam, tam, 3], y int [n]) con n_por_clase imágenes por clase, mezcladas."""
    rng = np.random.default_rng(semilla)
    X = np.stack([funcion(c, rng, tam) for c in clases for _ in range(n_por_clase)])
    y = np.repeat(np.arange(len(clases)), n_por_clase)
    orden = rng.permutation(len(y))
    return X[orden], y[orden]


PARECIDAS = {"hotel": "apartamentos", "apartamentos": "hotel", "casa_rural": "camping", "camping": "casa_rural", "piscina": "camping"}


def crear_dataset_carpetas(carpeta, n_por_clase=100, semilla=7, tam=48, ruido=8, prop_ambiguas=0.0):
    """Escribe PNG en carpeta/<clase>/img_000.png (estructura que espera image_dataset_from_directory).

    prop_ambiguas: fracción de imágenes cuyo dibujo corresponde a una clase parecida (PARECIDAS) pero se guarda
    en la carpeta de la otra: simula etiquetas dudosas o erróneas. Se anotan en carpeta/_ambiguas.csv (solo docente).
    """
    import os
    rng = np.random.default_rng(semilla)
    ambiguas = []
    for clase in CLASES_ALOJ:
        os.makedirs(os.path.join(carpeta, clase), exist_ok=True)
        for i in range(n_por_clase):
            dibujo = PARECIDAS[clase] if rng.random() < prop_ambiguas else clase
            if dibujo != clase:
                ambiguas.append(f"{clase}/img_{i:03d}.png,{dibujo}")
            Image.fromarray(dibujar_alojamiento(dibujo, rng, tam, ruido)).save(os.path.join(carpeta, clase, f"img_{i:03d}.png"))
    with open(os.path.join(carpeta, "_ambiguas.csv"), "w") as f:
        f.write("archivo,dibujo_real\n" + "\n".join(ambiguas) + "\n")
    return carpeta

### ✏️ 1 · Elegid el problema

Cambiad **PROBLEMA** por uno de estos nombres (la infografía explica cada uno):

| PROBLEMA | Qué reconoce | Imágenes |
|---|---|---|
| `digitos` | dígitos 0–9 | 8×8, reales (scikit-learn) |
| `fashion` | 10 prendas de ropa | 28×28, reales |
| `alojamientos` | hotel, apartamentos, casa rural, camping, piscina | 48×48, dibujadas por el curso |
| `formas` | 8 patrones y figuras | 48×48, dibujadas por el curso |
| `mnist` | dígitos escritos a mano 0–9 | 28×28, reales |
| `fashion_parecidas` | camiseta, jersey, abrigo, camisa | 28×28, reales (las 4 prendas que más se parecen) |

In [ ]:
PROBLEMA = "digitos"

### ⚙️ Preparar las imágenes — pulsa ▶

Tres partes: **entrenamiento** (aprende), **validación** (para elegir) y **prueba** (se mira una sola vez, al final).

In [ ]:
MAX_IMAGENES = 6000          # tope para que entrene en segundos

def cargar(problema):
    if problema == "digitos":
        from sklearn.datasets import load_digits
        d = load_digits(); return (d.images / 16.0)[..., None], d.target, [str(i) for i in range(10)]
    if problema == "fashion":
        (x, y), _ = keras.datasets.fashion_mnist.load_data()
        nombres = ["camiseta", "pantalón", "jersey", "vestido", "abrigo", "sandalia", "camisa", "zapatilla", "bolso", "botín"]
        return (x / 255.0)[..., None], y, nombres
    if problema == "alojamientos":
        X, y = generar_lote(dibujar_alojamiento, CLASES_ALOJ, 300); return X / 255.0, y, CLASES_ALOJ
    if problema == "formas":
        X, y = generar_lote(dibujar_forma, CLASES_FORMAS, 200); return X / 255.0, y, CLASES_FORMAS
    if problema == "mnist":
        (x, y), _ = keras.datasets.mnist.load_data(); return (x / 255.0)[..., None], y, [str(i) for i in range(10)]
    if problema == "fashion_parecidas":
        (x, y), _ = keras.datasets.fashion_mnist.load_data()
        elegidas = [0, 2, 4, 6]; m = np.isin(y, elegidas)
        return (x[m] / 255.0)[..., None], np.array([elegidas.index(v) for v in y[m]]), ["camiseta", "jersey", "abrigo", "camisa"]
    raise ValueError("PROBLEMA no válido: mira la tabla de arriba")

X, y, NOMBRES = cargar(PROBLEMA)
if len(X) > MAX_IMAGENES:
    X, _, y, _ = train_test_split(X, y, train_size=MAX_IMAGENES, random_state=SEMILLA, stratify=y)
X = X.astype("float32"); N_CLASES = len(NOMBRES)
X_res, X_pru, y_res, y_pru = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
X_ent, X_val, y_ent, y_val = train_test_split(X_res, y_res, test_size=0.25, random_state=SEMILLA, stratify=y_res)
LINEA_BASE = np.bincount(y_pru).max() / len(y_pru)
print(f"{len(X)} imágenes de {X.shape[1]}×{X.shape[2]} ({X.shape[3]} canal/es) · {N_CLASES} clases: {NOMBRES}")
print(f"LÍNEA BASE: sin red (diciendo siempre la clase más habitual) se acierta el {LINEA_BASE:.1%}. Vuestra red tiene que mejorarlo de verdad.")
fig, ejes = plt.subplots(2, 8, figsize=(12, 3.2))
for i, e in enumerate(ejes.ravel()):
    e.imshow(X_ent[i].squeeze(), cmap="gray_r" if X.shape[3] == 1 else None); e.set_title(NOMBRES[y_ent[i]], fontsize=8); e.axis("off")
plt.tight_layout(); plt.show()

### ⚙️ Las variantes de red — pulsa ▶

- **densa**: aplana la imagen y la mira entera, sin saber qué píxeles son vecinos.
- **cnn**: convoluciones + pooling: busca patrones pequeños donde estén.
- **cnn_aug**: la misma CNN pero viendo cada imagen un poco movida, girada y ampliada (*data augmentation* suave, sin espejos: un 6 espejado ya no es un 6).

In [ ]:
def construir(variante, filtros=16):
    capas = [keras.layers.Input(shape=X.shape[1:])]
    if variante == "cnn_aug":
        capas += [keras.layers.RandomTranslation(0.1, 0.1), keras.layers.RandomRotation(0.05), keras.layers.RandomZoom(0.1)]
    if variante == "densa":
        capas += [keras.layers.Flatten(), keras.layers.Dense(64, activation="relu")]
    else:
        capas += [keras.layers.Conv2D(filtros, 3, padding="same", activation="relu"), keras.layers.MaxPooling2D(),
                  keras.layers.Conv2D(filtros * 2, 3, padding="same", activation="relu"), keras.layers.MaxPooling2D(),
                  keras.layers.Flatten(), keras.layers.Dense(32, activation="relu")]
    capas += [keras.layers.Dense(N_CLASES, activation="softmax")]
    m = keras.Sequential(capas)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

def entrenar(variante, filtros=16, epocas=30):
    keras.utils.set_random_seed(SEMILLA)
    m = construir(variante, filtros)
    parar = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    h = m.fit(X_ent, y_ent, validation_data=(X_val, y_val), epochs=epocas, batch_size=64, callbacks=[parar], verbose=0)
    return m, h
print("Listo ✔")

### ✏️ 2 · Comparar variantes y elegir (solo con la validación)

Pulsad ▶ y mirad **val_acc** (más alto = mejor), **epocas** (si llega a 30, aún podía aprender) y **parametros** (cuánto pesa la red). Escribid en **ELEGIDA** la variante que justificáis. Extra 🔴: cambiad `FILTROS` y comparad.

In [ ]:
VARIANTES = ["densa", "cnn", "cnn_aug"]
FILTROS = 16
filas = []
for v in VARIANTES:
    t0 = time.time(); m, h = entrenar(v, FILTROS)
    filas.append(dict(variante=v, val_acc=round(max(h.history["val_accuracy"]), 3), epocas=len(h.history["loss"]),
                      parametros=m.count_params(), segundos=round(time.time() - t0)))
tabla = pd.DataFrame(filas); display(tabla)
ELEGIDA = "cnn"          # ✏️ vuestra variante, justificada con la tabla de arriba

### ⚙️ Entrenar la elegida y ver las curvas — pulsa ▶

¿Se separan las dos curvas? Si la de validación sube mientras la de entrenamiento baja, hay sobreajuste.

In [ ]:
modelo, hist = entrenar(ELEGIDA, FILTROS)
plt.plot(hist.history["loss"], label="entrenamiento"); plt.plot(hist.history["val_loss"], label="validación")
plt.xlabel("época"); plt.ylabel("pérdida"); plt.title(ELEGIDA); plt.legend(); plt.show()

### ✏️ 3 · La prueba, una sola vez, y dónde se equivoca

- **recall por clase**: de las imágenes de esa clase, cuántas reconoce. **La peor clase** cuenta más que la media.
- Mirad la **matriz de confusión** (filas = realidad, columnas = lo que dijo la red) y los **fallos**: ¿son errores razonables (cosas que también confundiría una persona) o raros?
- No volváis a cambiar nada después de ver estos números.

In [ ]:
p = modelo.predict(X_pru, verbose=0); c = p.argmax(1)
acc = accuracy_score(y_pru, c)
inf = pd.DataFrame(classification_report(y_pru, c, target_names=NOMBRES, output_dict=True, zero_division=0)).T.iloc[:N_CLASES]
display(inf[["precision", "recall", "f1-score", "support"]].round(2))
peor = inf["recall"].idxmin()
cm = confusion_matrix(y_pru, c); fuera = cm - np.diag(np.diag(cm))
r, k = np.unravel_index(fuera.argmax(), fuera.shape)
fig, ax = plt.subplots(figsize=(5, 4.5)); ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(N_CLASES)); ax.set_xticklabels(NOMBRES, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(N_CLASES)); ax.set_yticklabels(NOMBRES, fontsize=8); ax.set_xlabel("lo que dijo la red"); ax.set_ylabel("realidad")
for i in range(N_CLASES):
    for j in range(N_CLASES): ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7)
plt.tight_layout(); plt.show()
mal = np.where(c != y_pru)[0][:8]
fig, ejes = plt.subplots(1, max(len(mal), 1), figsize=(12, 2.4))
for e, i in zip(np.atleast_1d(ejes), mal):
    e.imshow(X_pru[i].squeeze(), cmap="gray_r" if X.shape[3] == 1 else None); e.set_title(f"real: {NOMBRES[y_pru[i]]}\nred: {NOMBRES[c[i]]}", fontsize=8); e.axis("off")
plt.suptitle("Algunos fallos"); plt.tight_layout(); plt.show()
print(f"""
================ FICHA PARA LA PRESENTACIÓN ================
Problema ........ reconocer «{PROBLEMA}» · {len(X)} imágenes de {X.shape[1]}×{X.shape[2]} · {N_CLASES} clases
Variantes ....... {[(f['variante'], f['val_acc']) for f in filas]}  (val_acc)
Elegida ......... {ELEGIDA} ({modelo.count_params()} parámetros)
--- PRUEBA (una sola vez) ---
Línea base ...... {LINEA_BASE:.1%}   (acertar sin red)
Exactitud ....... {acc:.1%}
Peor clase ...... {peor} (recall {inf.loc[peor, "recall"]:.1%})
Confusión mayor . {(NOMBRES[r] + " → " + NOMBRES[k] + "  (%d casos)" % fuera[r, k]) if fuera.max() > 0 else "ninguna: no hay fallos (¿es creíble un 100 %?)"}
Fallos totales .. {int((c != y_pru).sum())} de {len(y_pru)}
============================================================
""")

### ✏️ 4 · La aplicación: probar una imagen nueva

`probar(i)` coge la imagen número `i` de la prueba y dice qué ve la red y con cuánta seguridad. Probad varias, sobre todo las que fallan.

In [ ]:
def probar(i=0):
    q = modelo.predict(X_pru[i:i + 1], verbose=0)[0]; top = q.argsort()[::-1][:3]
    plt.imshow(X_pru[i].squeeze(), cmap="gray_r" if X.shape[3] == 1 else None); plt.axis("off"); plt.show()
    print(f"La red ve: «{NOMBRES[top[0]]}» ({q[top[0]]:.0%} de seguridad)   (realidad: «{NOMBRES[y_pru[i]]}»)")
    print("Otras opciones:", ", ".join(f"{NOMBRES[t]} {q[t]:.0%}" for t in top[1:]))

probar(0)

### 🎤 Para la presentación

Copiad la **FICHA**, la tabla de la celda ✏️ 2, la matriz de confusión y los fallos. Preparad una frase para cada pregunta: **¿mejora la línea base? · ¿por qué esa variante? · ¿dónde se equivoca y es razonable? · ¿la usaríamos de verdad?**